# 04 — Classification + SHAP
**Goal:** Train sector classifier locally, evaluate on held-out test set, explain predictions with SHAP.

The Databricks version runs 30 Hyperopt trials. This notebook shows the same logic + SHAP explainability.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/sample/startups_sample.csv')
print(f'Records: {len(df)} | Sectors: {df["sector"].nunique()}')

In [ ]:
# Features and labels
le = LabelEncoder()
y = le.fit_transform(df['sector'])

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2), sublinear_tf=True)
X = tfidf.fit_transform(df['description'])

# Stratified split — 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')

In [ ]:
# Train classifier
model = LogisticRegression(
    C=1.0, max_iter=500,
    class_weight='balanced',
    solver='lbfgs',
    multi_class='multinomial',
    random_state=42
)
model.fit(X_train, y_train)

# Evaluate on HELD-OUT test set
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
# Confusion matrix
fig, ax = plt.subplots(figsize=(12, 10))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=le.classes_,
    ax=ax, xticks_rotation=45
)
ax.set_title('Confusion Matrix — Sector Classifier', fontweight='bold')
plt.tight_layout()
plt.savefig('../data/sample/confusion_matrix.png', dpi=120)
plt.show()

In [ ]:
# SHAP — which words drive sector classification?
# Using LinearExplainer: works directly with sparse TF-IDF matrix
explainer = shap.LinearExplainer(model, X_train, feature_perturbation='interventional')
shap_values = explainer.shap_values(X_test)

feature_names = tfidf.get_feature_names_out()

# Summary plot for first class
plt.figure()
shap.summary_plot(
    shap_values[0], X_test,
    feature_names=feature_names,
    max_display=15,
    show=False
)
plt.title(f'SHAP — Top features for: {le.classes_[0]}', fontweight='bold')
plt.tight_layout()
plt.savefig('../data/sample/shap_summary.png', dpi=120)
plt.show()
print('SHAP plot saved')

In [ ]:
# Business framing — what this means
print('=== BUSINESS IMPACT ===')
print('A VC analyst manually classifying 50 deals/week = ~30 hours of repetitive triage.')
print('This classifier screens a deal in under 1 second.')
print('Human judgment is preserved for the shortlist — not the filter.')
print()
print('SHAP tells us WHY each prediction was made.')
print('That explainability is what makes this production-ready, not just a demo.')